In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import re
from bracket_score import compute_all_scores
from constants import ESPN_FILE, BRACKETS_JSON, BRACKETS_FEATHER, DATA_DIRECTORY

In [ ]:
# espn = pd.read_csv(ESPN_FILE)
# results_dict = {}
# for gender in ["mens", "womens"]:
#     most_recent = max(espn[espn['gender'] == gender]['forecast_date'])
#     results = espn[
#         (espn['gender'] == gender) &
#         (espn['forecast_date'] == most_recent) &
#         (espn['rd1_win'] == 1)
#     ].reset_index(drop=True)
#     results['seed'] = results['team_seed'].apply(lambda x: int(re.sub('\D', '', x)))
#     results_dict[gender] = results

results_dict = {}
for gender in ["mens", "womens"]:
    results = pd.read_excel(f'{DATA_DIRECTORY.format(gender=gender)}/results.xlsx')
    results['seed'] = results['team_seed'].replace("\D", "", regex=True)
    results_dict[gender] = results

In [4]:
results_dict['mens'].sort_values('rd7_win', ascending=False).head()

,team_id,team_name,team_seed,team_region,playin_flag,team_alive,rd1_win,rd2_win,rd3_win,rd4_win,rd5_win,rd6_win,rd7_win,win_odds,timestamp,seed
0,41,Connecticut,1,East,0,1,1,1,1,1,1,1,0.254744,2.92551,2024-03-24 00:35:23.130,1
50,2509,Purdue,1,Midwest,0,1,1,1,1,1,1,1,0.121166,7.25311,2024-03-24 00:35:23.130,1
36,245,Texas A&M,9,South,0,1,1,1,0,0,0,0,0.000000,161.005,2024-03-24 00:35:23.130,9
37,275,Wisconsin,5,South,0,0,1,0,0,0,0,0,0.000000,--,2024-03-24 00:35:23.130,5
38,256,James Madison,12,South,0,1,1,1,0,0,0,0,0.000000,1812.8,2024-03-24 00:35:23.130,12


In [5]:
results_dict["mens"].loc[0, 'rd7_win'] = 1
results_dict["mens"].loc[50, 'rd7_win'] = 0
results_dict["mens"].head()

,team_id,team_name,team_seed,team_region,playin_flag,team_alive,rd1_win,rd2_win,rd3_win,rd4_win,rd5_win,rd6_win,rd7_win,win_odds,timestamp,seed
0,41,Connecticut,1,East,0,1,1,1,1,1,1,1,1.0,2.92551,2024-03-24 00:35:23.130,1
1,56,Stetson,16,East,0,0,1,0,0,0,0,0,0.0,--,2024-03-24 00:35:23.130,16
2,2226,Florida Atlantic,8,East,0,0,1,0,0,0,0,0,0.0,--,2024-03-24 00:35:23.130,8
3,77,Northwestern,9,East,0,1,1,1,0,0,0,0,0.0,2616.53,2024-03-24 00:35:23.130,9
4,21,San Diego State,5,East,0,1,1,1,1,0,0,0,0.0,342.808,2024-03-24 00:35:23.130,5


In [18]:
%%time
gender = "womens"
compute_all_scores(results_dict[gender], BRACKETS_JSON.format(gender=gender), BRACKETS_FEATHER.format(gender=gender))

Completed 1000000: 6572.19
CPU times: user 7min 52s, sys: 2min 57s, total: 10min 50s
Wall time: 1h 52min 15s


In [7]:
%%time
gender = "mens"
compute_all_scores(results_dict[gender], BRACKETS_JSON.format(gender=gender), BRACKETS_FEATHER.format(gender=gender))

Completed 1000000: 6300.84
CPU times: user 7min 50s, sys: 2min 34s, total: 10min 25s
Wall time: 1h 47min 41s


In [6]:
%%time
gender = "mens"
compute_all_scores(
    results_dict[gender],
    f"{DATA_DIRECTORY.format(gender='mens')}/brackets_kenpom.json",
    f"{DATA_DIRECTORY.format(gender='mens')}/brackets_kenpom.feather",
)

Completed 1000000: 6316.72
CPU times: user 7min 47s, sys: 2min 39s, total: 10min 26s
Wall time: 1h 47min 56s


## Timing

In [86]:
from bracket_score import compute_score
from bracket_functions import bracket_objects
import timeit
from bracket_functions import num_specific_seed_in_specific_round, score_bracket
from itertools import islice

In [7]:
brackets = bracket_objects(open(BRACKETS_JSON.format(gender=gender), 'rb'))

In [ ]:
pd.DataFrame([
    compute_score(
        obj=obj,
        results=results,
        start_time=start_time,
    )
    for obj in islice(brackets, 20)
])

In [8]:
obj = next(brackets)

In [10]:
import re

start_time = timeit.default_timer()

bracket_name = list(obj)[0]
bracket_num = int(re.search('\d+', bracket_name).group(0))
if bracket_num % 10000 == 0:
    stop_time = timeit.default_timer()
    print("{}: {}".format(bracket_num, round(stop_time - start_time, 2)))

values = list(obj.values())
bracket = pd.read_json(values[0][0])
prob = float(values[0][1])

merged = pd.merge(results, bracket, how='left', left_on='team_name', right_on='64')

total_score, potential_score, num_correct = score_bracket(merged)

0: 0.0


In [71]:
import numpy as np

def check_specific(binary_bracket, seed_bracket, seed_num, round_num):
    return (binary_bracket[round_num] & seed_bracket[seed_num]).sum()

res = {
    f"num_{seed_num}_in_{round_num}": check_specific(
        binary_bracket=binary_bracket,
        seed_bracket=seed_bracket,
        seed_num=seed_num,
        round_num=round_num,
    )
    for seed_num in range(1, 17)
    for round_num in ["32", "16", "8", "4", "2", "1"]
}

In [72]:
df

,type,num_in_round,seed,round
0,num_1_in_32,4,1,32
1,num_1_in_16,4,1,16
2,num_1_in_8,3,1,8
3,num_1_in_4,3,1,4
4,num_2_in_32,4,2,32
...,...,...,...,...
59,num_15_in_4,0,15,4
60,num_16_in_32,0,16,32
61,num_16_in_16,0,16,16
62,num_16_in_8,0,16,8


In [82]:
df = pd.Series(res).reset_index().rename(columns={"index": "type", 0: "num_in_round"})
df = pd.concat((
    df,
    df["type"].str.split("_", expand=True)[[1, 3]].rename(columns={1: "seed", 3: "round"}).astype(int)
), axis=1)
{
    f"best_round_by_seed_{seed}": value
    for seed, value in df.loc[df["num_in_round"] > 0].groupby("seed")["round"].min().items()
}


{'best_round_by_seed_1': 1,
 'best_round_by_seed_2': 8,
 'best_round_by_seed_3': 2,
 'best_round_by_seed_4': 8,
 'best_round_by_seed_5': 32,
 'best_round_by_seed_6': 8,
 'best_round_by_seed_7': 16,
 'best_round_by_seed_8': 32,
 'best_round_by_seed_9': 32,
 'best_round_by_seed_11': 32,
 'best_round_by_seed_12': 32}

In [ ]:
    f"num_{seed_num}_in_{round_num}": num_specific_seed_in_specific_round(
        binary_bracket=binary_bracket,
        seed_bracket=seed_bracket,
        seed_num=seed_num,
        round_num=round_num,
    )
    for seed_num in range(1, 17)
    for round_num in ["32", "16", "8", "4", "2", "1"]
}

In [26]:
%load_ext line_profiler

In [38]:
%lprun -f compute_score compute_score(obj, results, timeit.default_timer())

Timer unit: 1e-06 s

Total time: 0.394054 s
File: /Users/Alex/src/bracket-predictions/bracket_score.py
Function: compute_score at line 15

Line #      Hits         Time  Per Hit   % Time  Line Contents
    15                                           def compute_score(obj, results, start_time):
    16                                               """Read saved bracket, merge results, compute scores, save to dict."""
    17         1          5.0      5.0      0.0      bracket_name = list(obj)[0]
    18         1         19.0     19.0      0.0      bracket_num = int(re.search('\d+', bracket_name).group(0))
    19         1          2.0      2.0      0.0      if bracket_num % 10000 == 0:
    20                                                   stop_time = timeit.default_timer()
    21                                                   print("{}: {}".format(bracket_num, round(stop_time - start_time, 2)))
    22                                           
    23         1          4.0      4

In [83]:
%lprun -f compute_score compute_score(obj, results, timeit.default_timer())

0: 0.0


Timer unit: 1e-06 s

Total time: 0.167749 s
File: /Users/Alex/src/bracket-predictions/bracket_score.py
Function: compute_score at line 14

Line #      Hits         Time  Per Hit   % Time  Line Contents
    14                                           def compute_score(obj, results, start_time):
    15                                               """Read saved bracket, merge results, compute scores, save to dict."""
    16         1          8.0      8.0      0.0      bracket_name = list(obj)[0]
    17         1         22.0     22.0      0.0      bracket_num = int(re.search('\d+', bracket_name).group(0))
    18         1          6.0      6.0      0.0      if bracket_num % 10000 == 0:
    19         1          5.0      5.0      0.0          stop_time = timeit.default_timer()
    20         1        245.0    245.0      0.1          print("{}: {}".format(bracket_num, round(stop_time - start_time, 2)))
    21                                           
    22         1          7.0      7

In [21]:
%lprun -f num_specific_seed_in_specific_round compute_score(bracket, results, timeit.default_timer())

Timer unit: 1e-06 s

Total time: 0.196727 s
File: /Users/Alex/src/bracket-predictions/bracket_functions.py
Function: num_specific_seed_in_specific_round at line 124

Line #      Hits         Time  Per Hit   % Time  Line Contents
   124                                           def num_specific_seed_in_specific_round(merged, seed_num, round_num):
   125                                               """e.g. How many 16 seeds made it to the round of 32?"""
   126        64      88029.0   1375.5     44.7      merged['seed'] = merged['team_seed'].apply(lambda x: int(re.sub('\D', '', x)))
   127        64      31419.0    490.9     16.0      binary_round = merged[round_num] != ''
   128        64      32520.0    508.1     16.5      seed_round = merged['seed'] == seed_num
   129        64      25564.0    399.4     13.0      score_vec = binary_round & seed_round
   130        64      19195.0    299.9      9.8      return score_vec.sum()

In [24]:
%lprun -f check_longest_by_seed compute_score(bracket, results, timeit.default_timer())

Timer unit: 1e-06 s

Total time: 0.141292 s
File: /Users/Alex/src/bracket-predictions/bracket_functions.py
Function: check_longest_by_seed at line 133

Line #      Hits         Time  Per Hit   % Time  Line Contents
   133                                           def check_longest_by_seed(merged, seed_num):
   134                                               """e.g. How far did a 16 seed go?"""
   135        16         15.0      0.9      0.0      points = 0
   136       112        167.0      1.5      0.1      for index, round in enumerate(['32', '16', '8', '4', '2', '1']):
   137        96      36220.0    377.3     25.6          binary_round = merged[round] != ''
   138        96      37474.0    390.4     26.5          seed_round = merged['seed'] == seed_num
   139        96      37828.0    394.0     26.8          score_vec = binary_round & seed_round
   140        96        242.0      2.5      0.2          pts_per_correct = 10 * (2**index)  # ESPN scoring
   141        96      29135.

## Hypothetical

In [6]:
results.loc[results['team_name'] == 'Baylor', 'rd5_win'] = 0
results.loc[results['team_name'] == 'Baylor', 'rd6_win'] = 0
results.loc[results['team_name'] == 'Baylor', 'rd7_win'] = 0
results.loc[results['team_name'] == 'Arkansas', 'rd5_win'] = 1
results.loc[results['team_name'] == 'Arkansas', 'rd6_win'] = 1
results.loc[results['team_name'] == 'Arkansas', 'rd7_win'] = 1

/Users/Alex/opt/anaconda3/envs/py3/lib/python3.8/site-packages/pandas/core/indexing.py:966: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.obj[item] = s


In [7]:
# TODO Fix paths
%%time
compute_all_scores(results, JSON_FILE, "sim_hypothetical.feather")

Completed 1000000: 2748.25


In [6]:
import json
import requests
raw = json.loads(open("https://gambit-api.fantasy.espn.com/apis/v1/propositions/?challengeId=257", 'r'))

FileNotFoundError: [Errno 2] No such file or directory: 'https://gambit-api.fantasy.espn.com/apis/v1/propositions/?challengeId=257'

In [11]:
import requests
res = requests.get("https://gambit-api.fantasy.espn.com/apis/v1/propositions/?challengeId=257").json()

In [13]:
res[-1]

{'active': False,
 'actualOutcomeIds': ['29e79851-e4b2-11ef-b0aa-77c6c7c47db6',
  '29e79853-e4b2-11ef-b0aa-77c6c7c47db6'],
 'allowsTieScore': False,
 'automated': True,
 'availableDate': 1738863933013,
 'bye': False,
 'cardType': 'DENSE',
 'challengeId': 257,
 'completeDate': 1742754714016,
 'correctOutcomes': ['29e79851-e4b2-11ef-b0aa-77c6c7c47db6'],
 'creationInfo': {'clientAddress': '10.164.72.16',
  'date': 1738863932586,
  'platform': 'chui-admin',
  'source': '{54D92334-E22C-46AF-9923-34E22CA6AFC9}'},
 'date': 1742746200000,
 'dateTBD': False,
 'deleted': False,
 'description': 'Who will win this matchup?',
 'display': True,
 'displayOrder': 5,
 'featured': False,
 'id': '29e79850-e4b2-11ef-b0aa-77c6c7c47db6',
 'lastUpdateInfo': {'clientAddress': None,
  'date': 1742754714016,
  'platform': 'gambit-processor',
  'source': 'PropositionAutomationTaskProcessor'},
 'mappings': [{'type': 'SPORT', 'value': 'basketball'},
  {'type': 'LEAGUE', 'value': 'mens-college-basketball'},
  {'typ